In [0]:
#Create a DataFrame with a future date and an impossibly old date
from pyspark.sql.functions import col, current_date, to_date

data = [(1, "2024-01-15"), (2, "2027-12-31"), (3, "1900-01-01")]
df = spark.createDataFrame(data, ["id", "order_date"])
df = df.withColumn("order_date", col("order_date").cast("date"))
df.display()

In [0]:
# Catch the future date
future_dates = df.filter(col("order_date") > current_date())
future_dates.display()

In [0]:
# Catch the impossibly old date
from pyspark.sql.functions import lit
impossible_old_date = df.filter(col("order_date") <= to_date(lit("1900-01-01")))
impossible_old_date.display()

In [0]:
# Cross column check - ship date should never be before order date
data_orders = [(1, "2024-01-10", "2024-01-15"),
               (2, "2024-02-01", "2024-01-28"),
               (3, "2024-03-05", "2024-03-05")]
df_orders = spark.createDataFrame(data_orders, ["id", "order_date", "ship_date"])
df_orders = df_orders.withColumn("order_date", col("order_date").cast("date")) \
                      .withColumn("ship_date", col("ship_date").cast("date"))

impossible_sequence = df_orders.filter(col("ship_date") < col("order_date"))
impossible_sequence.display()

In [0]:
#Mixed date formats — watch one silently become null
#Check: one of these two rows likely comes back null — that's the format mismatch.
from pyspark.sql.functions import expr
data_formats = [(1, "15/01/2024"), (2, "2024-01-15")]
df_bad = spark.createDataFrame(data_formats, ["id", "raw_date"])
df_bad = df_bad.withColumn("parsed_date", expr("try_cast(raw_date as date)"))
df_bad.display()

In [0]:
#Fix it with try_to_date() and an explicit format (tolerates format mismatches)
from pyspark.sql.functions import expr, col
df_bad2 = df_bad.withColumn("parsed_date_v2", expr("try_to_date(raw_date, 'dd/MM/yyyy')"))
df_bad2.display()

In [0]:
# Measure and Report
total = df_orders.count()
failures = df_orders.filter(col("ship_date") < col("order_date")).count()
print(f"Impossible date sequences: {failures} out of {total} ({failures/total*100:.1f}%)")